In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from tqdm import tqdm

In [2]:
def one_hot_encode(sequence):
    base_to_index = {'A':0, 'C':1, 'G':2, 'T':3}
    encoding = np.zeros((len(sequence), 4), dtype=np.float32)
    for i, base in enumerate(sequence.upper()):
        if base in base_to_index:
            encoding[i, base_to_index[base]] = 1.0
        else:
            encoding[i, :] = 0.25  # Handle N bases
    return encoding

In [3]:
windows = pd.read_csv('./ath_upstream_downstream_1.5k.bed', sep='\t', header=None)
windows.columns = ['chr', 'start', 'end', 'four', 'five', 'strand']

sequences = []
names = []
for record in SeqIO.parse('./ath_upstream_downstream_1.5k.fasta', "fasta"):
    sequences.append(str(record.seq))
    names.append(record.id)

In [5]:
logits = pd.read_csv('../../results/review/9_ap2_tf_family/genome_wide_prob.tsv', sep='\t')
logits.columns = ['A', 'C', 'G', 'T']
logits.head()

,A,C,G,T
0,0.000222,0.000337,0.998667,0.000774
1,0.000545,0.002378,0.000062,0.997015
2,0.706056,0.008997,0.256071,0.028876
3,0.749986,0.122822,0.027315,0.099876
4,0.102097,0.029505,0.727728,0.140669


In [6]:
resProb_list = []
resOneHot_list = []
insert_rows = pd.DataFrame(0.5, index=range(20), columns=logits.columns)
current_logit_pos = 0  # Track position in logits DataFrame

In [7]:
for idx, row in tqdm(windows.iterrows(), total=len(windows)):
    start_pos, end_pos = row['start'], row['end']
    width = end_pos - start_pos
    
    # Extract current logits
    curLogits = logits.iloc[current_logit_pos: current_logit_pos + width]
    current_logit_pos += width
    
    # Append insert_rows and process sequence
    curLogits = pd.concat([curLogits, insert_rows], axis=0)
    curSeq = sequences[idx] + 'N'*20
    encoded = one_hot_encode(curSeq)
    
    resProb_list.append(curLogits)
    resOneHot_list.append(encoded)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80153/80153 [00:16<00:00, 4848.74it/s]


In [8]:
# Concatenate all results
resProb = pd.concat(resProb_list, axis=0).reset_index(drop=True)
resOneHot = np.vstack(resOneHot_list)

In [9]:
resProb_array = resProb.to_numpy() - 0.25

In [10]:
resProb_array = resProb_array[np.newaxis, :, :]
resProb_array.shape

(1, 20571740, 4)

In [11]:
resOneHot.shape

(20571740, 4)

In [12]:
resOneHot = resOneHot[np.newaxis, :, :]
resOneHot.shape

(1, 20571740, 4)

In [13]:
import modiscolite

In [14]:
pos_patterns, neg_patterns = modiscolite.tfmodisco.TFMoDISco(
    hypothetical_contribs=resProb_array,
    one_hot=resOneHot,
    max_seqlets_per_metacluster=100_000,
    sliding_window_size=20,
    flank_size=5,
    verbose=True)

Using 43321 positive seqlets


IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


In [15]:
modiscolite.io.save_hdf5('genome_wide_gpn_modisco_results.h5', pos_patterns, neg_patterns, window_size = 20)